# Model Training - ChatKasir

- Nama: Achmad Rif'an
- Bagian: AI-1 (Model Architect)

# 1. Setup Google Colab dengan GPU

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd

# verifikasi versi
print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")

In [ ]:
# verifikasi GPU
gpus = tf.config.list_physical_devices('GPU')
print(f"\nGPU tersedia: {len(gpus) > 0}")

if gpus:
    # tampilkan detail GPU yang aktif agar terdokumentasi
    for gpu in gpus:
        print(f"Nama GPU    : {gpu.name}")

    # aktifkan memory growth - GPU tidak langsung mengambil semua VRAM
    # tapi mengalokasikan secara bertahap sesuai kebutuhan
    # ini mencegah crash "out of memory" di awal training
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("Memory growth: aktif")
else:
    print("PERINGATAN: GPU tidak aktif! Cek Runtime -> Change runtime type")

# 2. Data Loading v3

In [ ]:
# menyimpan url dataset ke dalam variabel
url_food = "https://drive.google.com/uc?id=1xpoFjqAT9K0uwzSpVADm_EfKqG7dxVUI"
url_slang = "https://drive.google.com/uc?id=1G14C1qcqOp06Xs1HFiorE3Us_LLtaBs7"
url_sintetis = "https://drive.google.com/uc?id=17lFTivPH4BXEd6zoa-qjpUohNmh1ELdo"

# membaca dataset
df_food = pd.read_csv(url_food)
df_slang = pd.read_csv(url_slang)
df_sintetis = pd.read_csv(url_sintetis)

In [ ]:
# memuat dataset food
print(f"Total dataset food: {len(df_food)} baris.")

print("\nPratinjau 5 baris pertama dari Dataset Food:")
display(df_food.head())

In [ ]:
# memuat dataset slang
print(f"Total dataset slang: {len(df_slang)} baris.")

print("\nPratinjau 5 baris pertama dari Dataset Slang:")
display(df_slang.head())

In [ ]:
# memuat dataset sintetis
print(f"Total dataset sintetis: {len(df_sintetis)} baris.")

# memuat dataset sintetis
print("\nPratinjau 5 baris pertama dari Dataset Sintetis:")
display(df_sintetis.head())

In [ ]:
# memeriksa tipe data dataset sintetis
print("Informasi Dataset Sintetis:")
df_sintetis.info()

# 3. Preprocessing & Ekstraksi Fitur v3

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("Ekstraksi Fitur (X) dan Target (y)")
X = df_sintetis['input_text'].astype(str).values

# target untuk ke-3 Output Head
y_product = df_sintetis['product'].astype(str).values
y_quantity = df_sintetis['quantity'].astype(float).values
y_price_satuan = df_sintetis['price_satuan'].astype(float).values

# normalisasi harga: dibagi 1000 agar loss seimbang dengan quantity
# nilai -1 (harga tidak diketahui) dibiarkan tetap -1
y_price_satuan = np.where(y_price_satuan != -1.0, y_price_satuan / 1000.0, -1.0)

print("Fitur dan target berhasil diekstraksi.")

In [ ]:
print("Label Encoding untuk Entitas Produk")
label_encoder = LabelEncoder()

# gabungkan seluruh nama makanan dari df_food dan df_sintetis untuk melatih encoder
# memastikan semua kosakata makanan dikenali model.
all_products = pd.concat([df_food['name'], df_sintetis['product']]).astype(str).unique()
label_encoder.fit(all_products)

# ubah target product dari string menjadi integer
y_product_encoded = label_encoder.transform(y_product)
y_quantity = df_sintetis['quantity'].astype(float).values
y_price_satuan = df_sintetis['price_satuan'].astype(float).values

# scaling [0, 1]
y_quantity_scaled = y_quantity / 10.0
y_price_scaled = np.where(y_price_satuan != -1.0, y_price_satuan / 100000.0, -1.0)

# total kelas produk untuk arsitektur layer Dense output
num_product_classes = len(label_encoder.classes_)
print(f"LabelEncoder berhasil dibuat. Total kelas produk (num_product_classes): {num_product_classes}")

In [ ]:
print("Membagi Data ke Training dan Validation (80:20)")

X_train, X_val, y_prod_train, y_prod_val, y_qty_train, y_qty_val, y_price_satuan_train, y_price_satuan_val = train_test_split(
    X, y_product_encoded, y_quantity_scaled, y_price_scaled, test_size=0.2, random_state=42
)

print(f"Data Train: {len(X_train)} baris")
print(f"Data Val  : {len(X_val)} baris")

In [ ]:
print("Tokenisasi dan Padding Teks")

# menggunakan hyperparameter dari eksperimen Minggu 1
VOCAB_SIZE = 10000
MAX_SEQ_LEN = 30

# membuat kamus dengan mengabaikan kata asing
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")

# fit hanya pada X_train agar tidak ada data leakage
tokenizer.fit_on_texts(X_train)

X_train_padded = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_SEQ_LEN, padding='post', truncating='post')
X_val_padded = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=MAX_SEQ_LEN, padding='post', truncating='post')

print(f"Tokenisasi & Padding selesai")
print(f"Dimensi X_train_padded : {X_train_padded.shape}")
print(f"Total kelas produk     : {num_product_classes}")

# 4. Arsitektur Model v3
Layer Embedding -> Layer Shared Bi-LSTM -> Layer Dense multi-output menggunakan TensorFlow Functional API

In [ ]:
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Dropout, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

# Membangun Arsitektur Model v3 ChatKasir

# hyperparameter dari eksplorasi dataset
EMBEDDING_DIM = 256
LSTM_UNITS = 128

# INPUT LAYER
# menerima array token angka dengan panjang tetap (MAX_SEQ_LEN = 30)
input_layer = Input(shape=(MAX_SEQ_LEN,), name="input_tokens")

# EMBEDDING LAYER
# mask_zero=True agar padding (0) otomatis diabaikan oleh LSTM
x = Embedding(
    input_dim=VOCAB_SIZE,
    output_dim=EMBEDDING_DIM,
    mask_zero=True,
    name="embedding"
)(input_layer)

# Stacked Bidirectional LSTM (2 Lapis)
# membaca kalimat dari kiri-ke-kanan dan kanan-ke-kiri sekaligus

# Layer 1: return_sequences=True agar outputnya bisa dibaca oleh LSTM Layer 2
x = Bidirectional(LSTM(LSTM_UNITS, return_sequences=True), name="bi_lstm_1")(x)
x = Dropout(0.3, name="dropout_1")(x)

# Layer 2: return_sequences=False untuk merangkum hasil akhirnya
x = Bidirectional(LSTM(64, return_sequences=False), name="bi_lstm_2")(x)

# DROPOUT LAYER
# mematikan 30% neuron secara acak untuk mencegah overfitting
x = Dropout(0.3, name="dropout_2")(x)

# SHARED DENSE LAYER
shared_dense = Dense(256, activation="relu", name="shared_dense")(x)

# OUTPUT HEADS (Multi-Output)
# Head 1: Product, klasifikasi multi-kelas (Softmax)
# dengan extra layer 512 neuron untuk mengatasi 18.558 kelas
product_features = Dense(512, activation="relu", name="product_extra_dense")(shared_dense)
out_product = Dense(num_product_classes, activation="softmax", name="product")(product_features)

# Head 2: Quantity
# regresi jumlah (ReLU memastikan tidak ada jumlah negatif)
# menebak angka 0.1 sampai 1.0
out_quantity = Dense(1, activation="relu", name="quantity")(shared_dense)

# Head 3: Price, regresi harga satuan
# menambah dense layer untuk mempelajari nominal sebelum mengeluarkan prediksi final
# menebak angka 0.0 sampai 1.0
price_features = Dense(128, activation="relu")(shared_dense)
price_features = Dense(64, activation="relu")(price_features)
out_price = Dense(1, activation="relu", name="price_satuan")(price_features)

# BUILD MODEL
# menyatukan semua komponen menjadi model
model = Model(
    inputs=input_layer,
    outputs=[out_product, out_quantity, out_price],
    name="ChatKasir_v3_Deeper_LSTM"
)

# ringkasan arsitektur
# jumlah parameter dan koneksi antar layer
model.summary()

# 5. Custom Loss Function, Optimizer, Metrik Akurasi v3

In [ ]:
# mendefinisikan Custom Loss (MaskedPriceLoss)
class MaskedPriceLoss(tf.keras.losses.Loss):
    """
    Custom Loss Function untuk output head 'price_price'.
    Mengecualikan baris dengan price_satuan = -1 (harga tidak diketahui)
    dari perhitungan loss agar model tidak belajar pola yang keliru.
    """
    def __init__(self, name="masked_price_loss"):
        super().__init__(name=name)
        self.NULL_INDICATOR = -1.0

    def call(self, y_true, y_pred):
      # pastikan tipe datanya sama
      y_true = tf.cast(y_true, tf.float32)
      y_pred = tf.cast(y_pred, tf.float32)

      # ubah (Batch,) menjadi (Batch, 1)
      y_true = tf.reshape(y_true, [-1, 1])

      # hitung loss
      mask = tf.cast(tf.not_equal(y_true, self.NULL_INDICATOR), dtype=tf.float32)
      squared_error = tf.square(y_true - y_pred)
      masked_error = squared_error * mask

      # hitung squared error untuk semua baris
      squared_error = tf.square(y_true - y_pred)

      # terapkan mask
      # baris null (-1) akan dikalikan 0 sehingga error-nya = 0
      masked_error = squared_error * mask

      # Rata-ratakan loss berdasarkan jumlah baris valid saja
      # tf.maximum untuk menghindari error pembagian dengan nol (Division by Zero)
      n_valid = tf.maximum(tf.reduce_sum(mask), 1.0)
      loss = tf.reduce_sum(masked_error) / n_valid

      return loss

print("MaskedPriceLoss berhasil didefinisikan")

In [ ]:
# Kompilasi Model

custom_adam = Adam(learning_rate=0.0005)

model.compile(
    optimizer=custom_adam,

    # loss function tiap output
    loss={
        # Head 1 (Product): Klasifikasi multi-kelas dengan label integer
        "product": tf.keras.losses.SparseCategoricalCrossentropy(),

        # Head 2 (Quantity): Regresi angka kontinu, menghukum error besar
        "quantity": tf.keras.losses.MeanSquaredError(),

        # Head 3 (Price Satuan): Regresi menggunakan custom loss
        "price_satuan": MaskedPriceLoss(),
    },

    # bobot tiap loss terhadap total loss
    loss_weights={
        "product": 1.0,
        "quantity": 3.0,
        "price_satuan": 5.0,
    },

    # metrik evaluasi untuk masing-masing output head
    metrics={
        "product": ['accuracy'],
        "quantity": ['mae'],  # Mean Absolute Error agar selisih mudah dibaca
        "price_satuan": ['mae'],  # MAE untuk harga satuan
    }
)

print("Model berhasil dikompilasi dan siap dilatih")

# 6. Callback dan Training Iterasi v3

In [ ]:
print("=== Memulai Training Iterasi 3 ===")

# turunkan LR sebesar 50% (factor=0.5)
# jika val_loss tidak turun selama 2 epoch
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5, verbose=1
)

# stop training jika val_loss tidak membaik selama 5 epoch
# lalu kembalikan bobot model terbaik
early_stopping = EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
)

# menetapkan hyperparameter
EPOCHS = 30
BATCH_SIZE = 64

history_v3 = model.fit(
    x=X_train_padded,
    y={
        "product": y_prod_train,  # target untuk klasifikasi produk
        "quantity": y_qty_train,  # target untuk regresi kuantitas
        "price_satuan": y_price_satuan_train  # target untuk regresi harga satuan
    },
    validation_data=(
        X_val_padded,
        {
            "product": y_prod_val,
            "quantity": y_qty_val,
            "price_satuan": y_price_satuan_val
        }
    ),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[reduce_lr, early_stopping]
)

# 7. Performa Training v3

In [ ]:
print("=== Pencatatan Performa Training 3 (VALIDATION SET) ===")

# mengambil hasil evaluasi (validation)
actual_epochs = len(history_v3.history['val_loss'])
last_epoch_idx = actual_epochs - 1

# ekstraksi metrik dari history
val_loss = history_v3.history['val_loss'][last_epoch_idx]
val_prod_acc = history_v3.history['val_product_accuracy'][last_epoch_idx]
val_qty_mae = history_v3.history['val_quantity_mae'][last_epoch_idx]
val_price_satuan_mae = history_v3.history['val_price_satuan_mae'][last_epoch_idx]

# kalkulasi total
total_akurasi = val_prod_acc
total_mae = val_qty_mae + val_price_satuan_mae

print(f"Total Validation Loss   : {val_loss:.4f}")
print(f"Total Akurasi (Produk)  : {total_akurasi*100:.2f}%")
print(f"Total MAE (Qty + Harga) : {total_mae:.4f} (pada skala [0, 1])")
print("\nRincian Error Regresi:")
print(f"  * MAE Quantity        : {val_qty_mae:.4f} (meleset ~{val_qty_mae * 10:.1f} porsi)")
print(f"  * MAE Price Satuan    : {val_price_satuan_mae:.4f} (meleset ~Rp {val_price_satuan_mae * 100000:,.0f})".replace(',', '.'))
print("\nIterasi training ke-3 selesai")

# 8. Curva Training Iterasi v3

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history):
    # membuat grid grafik 2x2
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Kurva Evaluasi Training Model v3 ChatKasir', fontsize=16, fontweight='bold')

    # Plot Total Loss
    axes[0, 0].plot(history.history['loss'], label='Train Loss', color='blue')
    axes[0, 0].plot(history.history['val_loss'], label='Validation Loss', color='orange')
    axes[0, 0].set_title('Total Loss (Gabungan 3 Head)')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, linestyle='--', alpha=0.7)

    # Plot Akurasi Product
    axes[0, 1].plot(history.history['product_accuracy'], label='Train Accuracy', color='green')
    axes[0, 1].plot(history.history['val_product_accuracy'], label='Validation Accuracy', color='red')
    axes[0, 1].set_title('Akurasi Klasifikasi: Product')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Akurasi')
    axes[0, 1].legend()
    axes[0, 1].grid(True, linestyle='--', alpha=0.7)

    # Plot MAE Quantity
    axes[1, 0].plot(history.history['quantity_mae'], label='Train MAE', color='purple')
    axes[1, 0].plot(history.history['val_quantity_mae'], label='Validation MAE', color='brown')
    axes[1, 0].set_title('Error Regresi: MAE Quantity')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Mean Absolute Error')
    axes[1, 0].legend()
    axes[1, 0].grid(True, linestyle='--', alpha=0.7)

    # Plot MAE Price Satuan
    axes[1, 1].plot(history.history['price_satuan_mae'], label='Train MAE', color='teal')
    axes[1, 1].plot(history.history['val_price_satuan_mae'], label='Validation MAE', color='magenta')
    axes[1, 1].set_title('Error Regresi: MAE Price Satuan (Skala x1000)')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Mean Absolute Error')
    axes[1, 1].legend()
    axes[1, 1].grid(True, linestyle='--', alpha=0.7)

    # merapikan jarak antar grafik
    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.show()

# panggil fungsi
plot_training_history(history_v3)

# 9. Analisis Error Model v3

In [ ]:
import pandas as pd
import numpy as np

print("Melakukan prediksi pada Validation Set...")

# prediksi dari model v3
predictions = model.predict(X_val_padded)

# output 0: Product (ambil indeks probabilitas tertinggi dengan argmax)
pred_product_idx = np.argmax(predictions[0], axis=1)
# output 1: Quantity (bulatkan karena jumlah pesanan pasti integer)
pred_quantity = np.round(predictions[1].flatten())
# output 2: Price Satuan (flatten array 2D ke 1D)
pred_price = predictions[2].flatten()

# kembalikan label produk ke bentuk teks (inverse transform)
true_product_labels = label_encoder.inverse_transform(y_prod_val)
pred_product_labels = label_encoder.inverse_transform(pred_product_idx)

# kembalikan skala (x10 untuk qty, x100.000 untuk harga)
true_quantity_real = y_qty_val * 10.0
pred_quantity_real = np.round(pred_quantity * 10.0)

true_price_real = np.where(y_price_satuan_val != -1.0, y_price_satuan_val * 100000.0, -1.0)
pred_price_real = pred_price * 100000.0

# buat dataframe untuk membandingkan prediksi dengan nilai asli
df_errors = pd.DataFrame({
    'teks_input': X_val, # teks chat mentah sebelum tokenisasi
    'true_product': true_product_labels,
    'pred_product': pred_product_labels,
    'true_quantity': y_qty_val,
    'pred_quantity': pred_quantity,
    'true_price': y_price_satuan_val,
    'pred_price': pred_price
})

# hitung Error
df_errors['product_match'] = df_errors['true_product'] == df_errors['pred_product']
df_errors['quantity_error'] = np.abs(df_errors['true_quantity'] - df_errors['pred_quantity'])

# untuk harga, abaikan (set error 0) baris yang harganya -1
df_errors['price_error'] = np.where(
    df_errors['true_price'] >= 0,
    np.abs(df_errors['true_price'] - df_errors['pred_price']),
    0
)

# tampilkan analisis kesalahan
print("\nIDENTIFIKASI POLA KEGAGALAN MODEL V3")

print("\nENTITAS PRODUK (Top 5 Kesalahan)")
salah_produk = df_errors[~df_errors['product_match']]

print(f"Total baris salah tebak produk: {len(salah_produk)} dari {len(df_errors)}")
display(salah_produk[['teks_input', 'true_product', 'pred_product']].head())

In [ ]:
print("\nENTITAS QUANTITY (Top 5 Error Terbesar)")
display(df_errors.sort_values(by='quantity_error', ascending=False)[['teks_input', 'true_quantity', 'pred_quantity', 'quantity_error']].head())

In [ ]:
print("\nENTITAS HARGA (Top 5 Error Terbesar)")
# Hanya tampilkan baris yang memang memiliki harga valid
display(df_errors[df_errors['true_price'] >= 0].sort_values(by='price_error', ascending=False)[['teks_input', 'true_price', 'pred_price', 'price_error']].head())